# 04 — Feature Selection

**Purpose:** Identify the most predictive features using three complementary methods,
then persist the final selected feature list for use by the modelling notebook.

Methods:
1. **Univariate log-rank** — survival-specific association test per feature
2. **Correlation-based** — remove highly redundant features
3. **Model-based importance** — Random Survival Forest importances
4. **Iterative elimination** — recursive feature elimination with RSF

Final output: consensus feature list based on a voting threshold.

---
**Inputs:** `data/processed/X_train.parquet`, `data/processed/y_train.parquet`  
**Outputs:** `data/processed/selected_features.json`

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines.statistics import logrank_test
from scipy import stats
from sklearn.inspection import permutation_importance
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

from src.data_utils import load_data, load_config, DATA_PROCESSED
from src.feature_engineering import get_feature_groups

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)
print('Ready.')

In [ ]:
cfg = load_config()
DURATION_COL = cfg['target']['duration_col']
EVENT_COL    = cfg['target']['event_col']
SEED         = cfg['experiment']['random_state']

X_train = load_data('X_train', stage='processed')
y_train = load_data('y_train', stage='processed')

duration_train = y_train[DURATION_COL]
event_train    = y_train[EVENT_COL]

print(f'X_train: {X_train.shape}')
print(f'Features: {X_train.columns.tolist()}')

## 1. Univariate Association (Log-Rank / Correlation)

For numeric features: Spearman correlation with duration.  
For categorical features: log-rank test across groups.

In [ ]:
univariate_scores = {}
cat_cols = [c for c in cfg['categorical_features'] if c in X_train.columns]
num_cols = [c for c in X_train.columns if c not in cat_cols]

# Numeric: Spearman |r| with duration as proxy
for col in num_cols:
    r, p = stats.spearmanr(X_train[col], duration_train)
    univariate_scores[col] = {'method': 'spearman', 'score': abs(r), 'p_value': p}

# Categorical: median duration difference across groups
for col in cat_cols:
    groups = X_train[col].unique()
    if len(groups) < 2:
        continue
    # Use variance ratio as a proxy score
    group_medians = X_train.groupby(col).apply(lambda g: duration_train.loc[g.index].median())
    score = group_medians.std() / (duration_train.std() + 1e-9)
    univariate_scores[col] = {'method': 'group_variance', 'score': float(score), 'p_value': np.nan}

uni_df = pd.DataFrame(univariate_scores).T.sort_values('score', ascending=False)
print('Univariate association scores:')
display(uni_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(4, len(uni_df) * 0.35)))
uni_df['score'][::-1].plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(0.05, color='tomato', linestyle='--', label='threshold = 0.05')
ax.set_title('Univariate Association Score')
ax.set_xlabel('|Spearman r| or group variance ratio')
ax.legend()
plt.tight_layout()
plt.show()

# Features passing univariate threshold
UNIVAR_THRESHOLD = 0.05
selected_univar = uni_df[uni_df['score'] >= UNIVAR_THRESHOLD].index.tolist()
print(f'\nSelected by univariate ({len(selected_univar)}): {selected_univar}')

## 2. Correlation-Based Redundancy Removal

In [ ]:
CORR_THRESHOLD = 0.85

corr_matrix = X_train[num_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))

# Remove the lower-importance member of each highly correlated pair
to_drop_corr = set()
for col in upper.columns:
    if any(upper[col] > CORR_THRESHOLD):
        # Keep the one with higher univariate score
        partners = upper.index[upper[col] > CORR_THRESHOLD].tolist()
        for partner in partners:
            score_col     = uni_df.loc[col, 'score']     if col     in uni_df.index else 0
            score_partner = uni_df.loc[partner, 'score'] if partner in uni_df.index else 0
            if score_col <= score_partner:
                to_drop_corr.add(col)
            else:
                to_drop_corr.add(partner)

selected_no_corr = [c for c in X_train.columns if c not in to_drop_corr]
print(f'Dropped due to high correlation ({len(to_drop_corr)}): {sorted(to_drop_corr)}')
print(f'Remaining ({len(selected_no_corr)}): {selected_no_corr}')

## 3. Model-Based Feature Importance (RSF)

In [ ]:
y_sksurv = Surv.from_arrays(event_train.astype(bool).values, duration_train.values)

rsf_sel = RandomSurvivalForest(
    n_estimators=100,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=SEED,
    n_jobs=-1,
)
rsf_sel.fit(X_train[selected_no_corr].values, y_sksurv)

importance_df = pd.Series(
    rsf_sel.feature_importances_,
    index=selected_no_corr,
).sort_values(ascending=False)

print('RSF Feature Importances:')
display(importance_df.round(4).to_frame('importance'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(4, len(importance_df) * 0.35)))
importance_df[::-1].plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(importance_df.mean(), color='tomato', linestyle='--', label='mean importance')
ax.set_title('RSF Feature Importances')
ax.legend()
plt.tight_layout()
plt.show()

# Select features above mean importance
mean_imp = importance_df.mean()
selected_rsf = importance_df[importance_df >= mean_imp * 0.5].index.tolist()
print(f'\nSelected by RSF importance ({len(selected_rsf)}): {selected_rsf}')

## 4. Consensus Selection

A feature is selected if it appears in **≥ 2 of 3** methods:
univariate, no-redundancy, and model-importance.

In [ ]:
all_features = list(X_train.columns)
vote_counts = pd.Series(0, index=all_features)

for f in selected_univar:
    if f in vote_counts.index:
        vote_counts[f] += 1
for f in selected_no_corr:
    if f in vote_counts.index:
        vote_counts[f] += 1
for f in selected_rsf:
    if f in vote_counts.index:
        vote_counts[f] += 1

VOTE_THRESHOLD = 2
selected_final = vote_counts[vote_counts >= VOTE_THRESHOLD].sort_values(ascending=False).index.tolist()

print(f'Votes per feature:')
display(vote_counts.sort_values(ascending=False).to_frame('votes'))
print(f'\nFinal selected features ({len(selected_final)}):')
print(selected_final)

In [ ]:
# Visualise vote counts
fig, ax = plt.subplots(figsize=(8, max(4, len(vote_counts) * 0.35)))
colors = ['#4CAF50' if v >= VOTE_THRESHOLD else '#9E9E9E' for v in vote_counts.sort_values().values]
vote_counts.sort_values().plot(kind='barh', ax=ax, color=colors)
ax.axvline(VOTE_THRESHOLD, color='tomato', linestyle='--', label=f'threshold = {VOTE_THRESHOLD}')
ax.set_title('Feature Selection Vote Count  (green = selected)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Save Selected Features

In [ ]:
output = {
    'selected_features': selected_final,
    'n_selected': len(selected_final),
    'n_total': len(all_features),
    'vote_threshold': VOTE_THRESHOLD,
    'method_counts': {
        'univariate': len(selected_univar),
        'no_redundancy': len(selected_no_corr),
        'rsf_importance': len(selected_rsf),
    },
}

out_path = DATA_PROCESSED / 'selected_features.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\nSelected features saved → {out_path}')
print(f'Selected {len(selected_final)} / {len(all_features)} features.')
print('\n✓ Done. Proceed to 05_modeling_survival.ipynb')